In [1]:
"""
Example script to run the electricity model.
It consists of three parts:
1. Generation
2. Grid
(3. Storage, not yet implemented)
"""

import pandas as pd
import numpy as np
import xarray as xr
import scipy
import warnings
from pathlib import Path
import warnings
from pint.errors import UnitStrippedWarning
warnings.simplefilter("ignore", UnitStrippedWarning)


from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid,
    get_preprocessing_data_stor
)
# from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks, Maintenance, MaterialIntensities, SharesInflowStocks
from imagematerials.factory import ModelFactory, Sector
import prism

VARIANT = "VLHO"
SCEN = "SSP2"
scen_folder = SCEN + "_" + VARIANT
# path_base = Path().resolve() # TODO absolute path of file "preprocessing.py" ? current solution can differ depending on IDE used (?) 
path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials

# create the folder out_test if it does not exist
if not (path_base / 'imagematerials' / 'electricity' / 'out_test').is_dir():
    (path_base / 'imagematerials' / 'electricity' / 'out_test').mkdir(parents=True)

YEAR_START = 1971   # start year of the simulation period
YEAR_END = 2100     # end year of the calculations
YEAR_OUT = 2100     # year of output generation = last year of reporting

# Generation -------------------------------------------------------

In [2]:
print("Base path:", path_base)
YEAR_START = 1971  # start year of the simulation period
YEAR_END = 2100    # end year of the calculations
YEAR_OUT = 2100    # year of output generation = last year of reporting
prep_data = get_preprocessing_data_gen(path_base, SCEN, VARIANT, YEAR_START, YEAR_END, YEAR_OUT)

Base path: C:\Users\Judit\PhD\Coding\image-materials
Path to image output: C:\Users\Judit\PhD\Coding\image-materials\data\raw\image\SSP2_VLHO\EnergyServices
gcap_stock to xarray Dataset
gcap_types_materials to xarray Dataset
gcap_lifetime_distr not in conversion_table


In [ ]:
# Define the complete timeline, including historic tail
time_start = prep_data["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

sec_electr_gen = Sector("electr_gen", prep_data)

factory = ModelFactory(
    sec_electr_gen, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model = factory.finish()

model.simulate(simulation_timeline)

list(model.electr_gen)

['gcap_lifetime_distr',
 'stocks',
 'material_intensities',
 'lifetimes',
 'knowledge_graph',
 'set_unit_flexible',
 'outflow_by_cohort',
 'inflow',
 'stock_by_cohort',
 'stock_by_cohort_materials',
 'inflow_materials',
 'outflow_by_cohort_materials']

# Grid -------------------------------------------------------------

In [3]:
YEAR_START = 1971  # start year of the simulation period
YEAR_END = 2100    # end year of the calculations
YEAR_OUT = 2100    # year of output generation = last year of reporting
prep_data_lines, prep_data_add = get_preprocessing_data_grid(path_base, SCEN, VARIANT, YEAR_START, YEAR_END, YEAR_OUT)

grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table


## Lines ----

In [5]:
# Define the complete timeline, including historic tail
time_start = prep_data_lines["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

sec_electr_grid_lines = Sector("electr_grid_lines", prep_data_lines)

factory = ModelFactory(
    sec_electr_grid_lines, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_lines = factory.finish()

model_lines.simulate(simulation_timeline)
list(model_lines.electr_grid_lines)

['lifetime_grid_distr',
 'stocks',
 'material_intensities',
 'lifetimes',
 'knowledge_graph',
 'set_unit_flexible',
 'outflow_by_cohort',
 'inflow',
 'stock_by_cohort',
 'stock_by_cohort_materials',
 'inflow_materials',
 'outflow_by_cohort_materials']

In [ ]:
model_lines.electr_grid_lines["stocks"]

TimeVariable(
  timeline=Timeline(start=1926, end=2100, stepsize=1),
  unit=kilometer,
  dims=
	Dimension(label="Region", coords=[np.str_('CAN'), np.str_('USA'),
	  np.str_('MEX'), np.str_('RCAM'), np.str_('BRA'), np.str_('RSAM'),
	  np.str_('NAF'), np.str_('WAF'), np.str_('EAF'), np.str_('SAF'),
	  np.str_('WEU'), np.str_('CEU'), np.str_('TUR'), np.str_('UKR'),
	  np.str_('STAN'), np.str_('RUS'), np.str_('ME'), np.str_('INDIA'),
	  np.str_('KOR'), np.str_('CHN'), np.str_('SEAS'), np.str_('INDO'),
	  np.str_('JAP'), np.str_('OCE'), np.str_('RSAS'), np.str_('RSAF')])
	  Dimension(label="Type", coords=[np.str_('HV - Lines - Overhead'),
	  np.str_('HV - Lines - Underground'), np.str_('LV - Lines - Overhead'),
	  np.str_('LV - Lines - Underground'), np.str_('MV - Lines - Overhead'),
	  np.str_('MV - Lines - Underground')]),
  values=
	[[[0.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00]
	  [0.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00]
	  [0.0000e+00 0

## Grid Additions ----

In [ ]:
time_start = prep_data_add["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

sec_electr_grid_add = Sector("electr_grid_add", prep_data_add)

factory = ModelFactory(
    sec_electr_grid_add, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_add = factory.finish()

model_add.simulate(simulation_timeline)
list(model_add.electr_grid_add)

['lifetime_grid_distr',
 'stocks',
 'material_intensities',
 'lifetimes',
 'knowledge_graph',
 'set_unit_flexible',
 'outflow_by_cohort',
 'inflow',
 'stock_by_cohort',
 'stock_by_cohort_materials',
 'inflow_materials',
 'outflow_by_cohort_materials']

# Storage -------------------------------------------------------

In [2]:
YEAR_FIRST_STOR = 1907 # first use of pumped storage was in 1907 at the Engeweiher pumped storage facility near Schaffhausen, Switzerland (Mitali et al. 2022)
YEAR_START = 1971  # start year of the simulation period
YEAR_END = 2100    # end year of the calculations
YEAR_OUT = 2100    # year of output generation = last year of reporting

prep_data_phs, prep_data_oth_storage = get_preprocessing_data_stor(path_base, SCEN, VARIANT, YEAR_START, YEAR_END, YEAR_OUT)

Path to image output: C:\Users\Judit\PhD\Coding\image-materials\data\raw\image\SSP2_VLHO\EnergyServices
phs_stock to xarray Dataset
phs_materials to xarray Dataset
phs_lifetime_distr not in conversion_table
oth_storage_stock to xarray Dataset
oth_storage_lifetime_distr not in conversion_table
oth_storage_shares to xarray Dataset


In [3]:
# Pumped Hydropower Storage (PHS) =====

# # Define the complete timeline, including historic tail
time_start = prep_data_phs["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1) #1970

sec_electr_stor_phs = Sector("electr_stor_phs", prep_data_phs)

factory = ModelFactory(
    sec_electr_stor_phs, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_phs = factory.finish()

model_phs.simulate(simulation_timeline)
list(model_phs.electr_stor_phs)

['phs_lifetime_distr',
 'stocks',
 'material_intensities',
 'lifetimes',
 'knowledge_graph',
 'set_unit_flexible',
 'outflow_by_cohort',
 'inflow',
 'stock_by_cohort',
 'stock_by_cohort_materials',
 'inflow_materials',
 'outflow_by_cohort_materials']

In [5]:
# Other Storage =====

time_start = prep_data_oth_storage["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1) #1970

sec_electr_stor_oth = Sector("electr_stor_oth", prep_data_oth_storage, check_coordinates=False)

factory = ModelFactory(
    sec_electr_stor_oth, complete_timeline
    ).add(SharesInflowStocks
    ).add(MaterialIntensities
    )
model_oth = factory.finish()

model_oth.simulate(simulation_timeline)
list(model_oth.electr_stor_oth)

['oth_storage_lifetime_distr',
 'stocks',
 'material_intensities',
 'lifetimes',
 'shares',
 'knowledge_graph',
 'set_unit_flexible',
 'outflow_by_cohort',
 'inflow',
 'stock_by_cohort',
 'stock_by_cohort_materials',
 'inflow_materials',
 'outflow_by_cohort_materials']